In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA02 - Travel Lodging Entertainment_POs
   
   BUSINESS QUESTION: 
   During the assessment period, did the unit (i) provide or receive gifts, or anything 
   of value to/from OR (ii) pay for the travel, hospitality or entertainment of POs, 
   their family members or close associates?
   
   LOGIC SUMMARY:
   1. TARGET CATEGORY CODES: Grabs the full list of flagged Category Codes from the 
      ABAC category_codes_database.
   2. COUPA DATA ONLY: Unions the 7 Coupa files. Finance files are excluded for EBA02.
   3. PUBLIC OFFICIAL FILTER: Filters strictly for rows where the `PublicOfficial` 
      column equals 'Y' or 'YES'.
   4. ACCOUNT PARSING: Parses the 'Account' string (format: xxxx-xxxx-xxxxxx) to 
      extract the Cost Center (first block) and Category Code (last block). Uses 
      TRY_CAST to INT to safely strip leading zeros.
   5. FILTERING: INNER JOINS the parsed Coupa data to the ABAC Category Codes list 
      so we strictly only flag true T&E expenses for Public Officials.
   6. MAPPING & OUTPUT: Joins the standardized Cost Center Mapping Bootstrap to the 
      flagged transactions. Rolls up by AU_ID. Outputs 'Yes' if a match is found.
=================================================================================== */

WITH Target_Category_Codes AS (
    -- 1. Grab the ABAC category codes to filter for actual T&E expenses
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files into one master dataset
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_PO_Transactions AS (
    -- 3 & 4. Parse the Account string and STRICTLY filter for Public Officials
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- Catches 'Y' or 'YES'
      AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
),

Flagged_PO_Transactions AS (
    -- 5. Filter the PO transactions against the ABAC Category list
    SELECT 
        c.Cleaned_CC,
        c.CatCode
    FROM Coupa_PO_Transactions c
    INNER JOIN Target_Category_Codes t
        ON c.CatCode = t.CatCode
    WHERE c.Cleaned_CC IS NOT NULL
),

Mapped_AUs AS (
    -- 6a. Join the flagged PO transactions to the standardized CC Mapping
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        CASE WHEN f.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END AS Has_PO_Transaction
    FROM vw_cost_center_mapping_bootstrap m
    
    LEFT JOIN Flagged_PO_Transactions f
        ON CAST(m.Cost_Center_ID AS INT) = f.Cleaned_CC
)

-- 6b. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EBA02' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    CASE 
        WHEN SUM(Has_PO_Transaction) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA02 - Public Official Transactions Drill-Down
   
   PURPOSE: 
   Provides a row-by-row view of every Cost Center that triggered a 'Yes' for EBA02.
   Shows the raw account string, the extracted Category Code, the Category Description, 
   and the Public Official flag to verify the parsing logic worked successfully.
=================================================================================== */

WITH Target_Category_Codes AS (
    SELECT DISTINCT TRIM(CatCode) AS CatCode, Description, Purpose
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_PO_Transactions AS (
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial_Flag,
        Account AS Raw_Account_String
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    c.Raw_Account_String,
    c.CatCode AS Extracted_Category_Code,
    t.Description AS ABAC_Category_Description,
    c.PublicOfficial_Flag
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN to strictly only show mapped units that actually had a PO transaction
INNER JOIN Coupa_PO_Transactions c
    ON CAST(m.Cost_Center_ID AS INT) = c.Cleaned_CC
    
-- INNER JOIN to ensure the Category Code matched the ABAC risk list
INNER JOIN Target_Category_Codes t
    ON c.CatCode = t.CatCode
    
ORDER BY BusinessID;